# MLflow Autologging: Autologging multiple models

This notebook is divided into small cells with markdown explanation and line-by-line comments. It uses `datasets/customer_churn_500.csv` from this folder.

## 1. Import required libraries

This cell imports MLflow, pandas, NumPy, matplotlib, and scikit-learn utilities.

In [ ]:
# Import os to create local folders and manage paths.
import os

# Import warnings to hide non-critical warning messages during training.
import warnings

# Ignore warnings for cleaner notebook output.
warnings.filterwarnings("ignore")

# Import MLflow for experiment tracking and model management.
import mlflow

# Import MLflow sklearn integration for logging sklearn models.
import mlflow.sklearn

# Import pandas for reading and processing CSV data.
import pandas as pd

# Import numpy for numerical calculations.
import numpy as np

# Import matplotlib for creating charts and artifact images.
import matplotlib.pyplot as plt

# Import train_test_split for splitting data into train and test sets.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for separate numerical and categorical processing.
from sklearn.compose import ColumnTransformer

# Import encoders and scalers for preprocessing.
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Import Pipeline to combine preprocessing and model into one object.
from sklearn.pipeline import Pipeline

# Import ML models.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Import regression metrics.
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 2. Configure MLflow

This cell configures MLflow to store runs locally in the current folder under `mlruns`.

In [ ]:
# Create local MLflow tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Store tracking data locally in this folder.
mlflow.set_tracking_uri("file:./mlruns")

# Set experiment name for this notebook.
mlflow.set_experiment("07_mlflow_autologging_03_autologging_multiple_models")

# Define the dataset path used by this notebook.
DATA_PATH = "datasets/customer_churn_500.csv"

# Print confirmation.
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Dataset path:", DATA_PATH)

## 3. Load and inspect dataset

The dataset contains 500 realtime-style customer records with categorical and numerical columns.

In [ ]:
# Read the CSV dataset from the local datasets folder.
df = pd.read_csv(DATA_PATH)

# Display the first five records.
display(df.head())

# Display dataset shape.
print("Dataset shape:", df.shape)

# Display column names.
print("Columns:", df.columns.tolist())

## 4. Prepare features and target

For classification, we predict `churn`. For regression, we predict `monthly_charges`.

In [ ]:
# Define classification target column.
target_column = "churn"

# Remove customer ID and target column from features.
X = df.drop(columns=["customer_id", target_column])

# Store the classification target.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Print feature summary.
print("Target column:", target_column)
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)

## 5. Build preprocessing pipeline

Categorical columns are one-hot encoded. Numerical columns are scaled.

In [ ]:
# Create preprocessing logic for numerical and categorical columns.
preprocessor = ColumnTransformer(
    transformers=[
        # Scale numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # Convert categorical columns into numeric one-hot encoded columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Print the preprocessing object.
preprocessor

## 6. Split the dataset

Training data is used to train the model. Testing data is used to evaluate the model.

In [ ]:
# Split data into training and testing datasets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print split sizes.
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 7. Enable autologging

Autologging captures model parameters, metrics, and artifacts automatically.

In [ ]:
# Enable MLflow autologging for scikit-learn.
mlflow.sklearn.autolog()

# Print confirmation.
print("MLflow sklearn autologging is enabled.")

## 7. Train multiple models and log each run

Each model configuration creates a separate MLflow run.

In [ ]:
# Define multiple model configurations for comparison.
model_configs = [
    ("logistic_regression", LogisticRegression(max_iter=1000)),
    ("random_forest_depth_4", RandomForestClassifier(n_estimators=120, max_depth=4, random_state=42)),
    ("random_forest_depth_8", RandomForestClassifier(n_estimators=180, max_depth=8, random_state=42)),
]

# Create an empty list to collect run results.
results = []

# Loop through every model configuration.
for model_name, estimator in model_configs:

    # Build preprocessing plus model pipeline.
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])

    # Start a separate MLflow run for this model.
    with mlflow.start_run(run_name=model_name):

        # Log model name.
        mlflow.log_param("model_name", model_name)

        # Log dataset path.
        mlflow.log_param("dataset", DATA_PATH)

        # Train model pipeline.
        pipeline.fit(X_train, y_train)

        # Predict on test data.
        predictions = pipeline.predict(X_test)

        # Calculate metrics.
        accuracy = accuracy_score(y_test, predictions)
        precision = precision_score(y_test, predictions, zero_division=0)
        recall = recall_score(y_test, predictions, zero_division=0)
        f1 = f1_score(y_test, predictions, zero_division=0)

        # Log all metrics.
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        # Log model artifact.
        mlflow.sklearn.log_model(pipeline, "model")

        # Append result for notebook display.
        results.append({
            "model_name": model_name,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
        })

# Display comparison table.
pd.DataFrame(results).sort_values("f1_score", ascending=False)

## Search runs programmatically

This cell shows how to read MLflow run history using code.

In [ ]:
# Search runs from the current experiment.
runs = mlflow.search_runs(experiment_names=["07_mlflow_autologging_03_autologging_multiple_models"])

# Display selected columns if available.
display(runs.head())

## Final trainer note

Ask students to open MLflow UI from this folder using:

```bash
mlflow ui --backend-store-uri ./mlruns
```

Then open `http://127.0.0.1:5000`.